# VLM-Anomaly — PatchCore Full MVTec Sweep (Kaggle GPU)

**Model:** PatchCore via [Anomalib](https://github.com/openvinotoolkit/anomalib) (Intel, Apache 2.0)  
**GPU:** T4 / P100 via Kaggle free tier (30 h/week)  
**Cost:** $0 — fully open-source classical baseline  
**Expected time:** ~45–75 min on T4 (vs 10+ h on CPU)  

## How to use this notebook
1. On Kaggle: **Settings → Accelerator → GPU T4 x2** (or P100)
2. Add the [MVTec AD dataset](https://www.kaggle.com/datasets/ipythonx/mvtec-ad) via **Add Data**
3. Run all cells
4. Download `patchcore_mvtec_results.json` from the **Output** tab
5. Drop the file into your local `results/` folder and run the report cell

## Local integration (after download)
```bash
# 1. Copy downloaded file
cp ~/Downloads/patchcore_mvtec_results.json results/

# 2. Regenerate report (run Cell 8 of any local notebook)
# OR from Python:
# from vlm_anomaly.analysis.report_generator import generate
# generate('results/', 'REPORT.md')

# 3. Commit
# git add results/patchcore_mvtec_results.json REPORT.md
# git commit -m 'results(classical/patchcore): MVTec sweep via Kaggle T4'
```


In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────────
import subprocess, sys

pkgs = [
    'anomalib==2.4.2',
    'timm>=1.0.0',
    'structlog>=24.4.0',
    'duckdb>=1.1.0',
    'scikit-learn>=1.6.0',
    'pydantic>=2.10.0',
    'pydantic-settings>=2.6.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
print('Dependencies installed.')


In [ ]:
# ── Cell 2: Clone repo ───────────────────────────────────────────────────────
import subprocess
from pathlib import Path

REPO_DIR = Path('/kaggle/working/VLM-Anomaly')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/sabareeswarans11/VLM-Anomaly.git',
         str(REPO_DIR)],
        check=True
    )
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

import sys
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

import vlm_anomaly
print(f'vlm_anomaly {vlm_anomaly.__version__} loaded from {REPO_DIR}')


In [ ]:
# ── Cell 3: Verify torch + CUDA + anomalib ──────────────────────────────────
import torch, anomalib

print(f'torch    : {torch.__version__}')
print(f'anomalib : {anomalib.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected — switch to GPU in Settings → Accelerator')


In [ ]:
# ── Cell 4: Find MVTec dataset ───────────────────────────────────────────────
# Supports common Kaggle MVTec dataset slugs — add the dataset via Add Data
# and this cell auto-detects the path.
from pathlib import Path

MVTEC_ROOT = None
CANDIDATES = [
    # Common Kaggle dataset slugs for MVTec AD
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad'),
    Path('/kaggle/input/mvtec-anomaly-detection'),
    Path('/kaggle/input/mvtecad'),
    Path('/kaggle/input/mvtec'),
]
EXPECTED_CATS = {'bottle', 'cable', 'capsule', 'carpet', 'grid'}

for c in CANDIDATES:
    if c.exists():
        found = {d.name for d in c.iterdir() if d.is_dir()}
        if EXPECTED_CATS.issubset(found):
            MVTEC_ROOT = c
            break

assert MVTEC_ROOT, (
    'MVTec dataset not found. Add it via Add Data.\n'
    f'Searched: {[str(c) for c in CANDIDATES]}'
)

categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f'MVTec root : {MVTEC_ROOT}')
print(f'Categories : {categories}')
print(f'Count      : {len(categories)}')


In [ ]:
# ── Cell 5: Configure ────────────────────────────────────────────────────────
import torch
from pathlib import Path

MODEL_NAME  = 'patchcore'
MODEL_ID    = f'classical/{MODEL_NAME}'
IMAGE_SIZE  = 256
DATASET     = 'mvtec'

# Use Lightning's authoritative check (not torch.backends.mps which can
# mislead on Intel Macs with AMD GPUs).
from vlm_anomaly.evaluators.classical_evaluator import best_accelerator
ACCELERATOR = best_accelerator()

# Kaggle working dirs
RESULTS_DIR = Path('/kaggle/working/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Symlink: Settings expects data_dir/mvtec/{category}/
DATA_DIR = Path('/kaggle/working/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
mvtec_link = DATA_DIR / 'mvtec'
if not mvtec_link.exists():
    mvtec_link.symlink_to(MVTEC_ROOT)

print(f'Model       : {MODEL_ID}')
print(f'Accelerator : {ACCELERATOR}')
print(f'Results dir : {RESULTS_DIR}')
print(f'Data dir    : {DATA_DIR} (mvtec → {MVTEC_ROOT})')


In [ ]:
# ── Cell 6: Build shared objects ─────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.logging import configure_logging
from vlm_anomaly.evaluators.classical_evaluator import ClassicalEvaluator

configure_logging(log_level='INFO')

settings = Settings(
    data_dir=str(DATA_DIR),
    results_dir=str(RESULTS_DIR),
)
print(f'Settings ready — data_dir={settings.data_dir}')


In [ ]:
# ── Cell 7: Run all 15 categories (idempotent, resumable) ────────────────────
import json as _json, time
from tqdm.auto import tqdm

all_results = []


def _already_done(cat):
    for f in RESULTS_DIR.glob(f'*_mvtec_{cat}_patchcore.json'):
        try:
            data = _json.loads(f.read_text())
            rows = data if isinstance(data, list) else [data]
            if any(r.get('model_id') == MODEL_ID for r in rows):
                return rows[0]
        except Exception:
            pass
    return None


for category in tqdm(categories, desc=f'PatchCore [{ACCELERATOR}]'):
    done = _already_done(category)
    if done:
        print(f'  [skip] {category} — already done')
        all_results.append(done)
        continue

    t0 = time.perf_counter()
    ev = ClassicalEvaluator(
        model_name=MODEL_NAME, dataset_name=DATASET,
        category=category, image_size=IMAGE_SIZE,
        accelerator=ACCELERATOR, settings=settings,
    )
    result = ev.run()
    elapsed = time.perf_counter() - t0
    all_results.append(result.model_dump())
    print(
        f'  {category:12s}  AUROC={result.auroc:.3f}  '
        f'F1={result.f1:.3f}  elapsed={elapsed:.0f}s  [{ACCELERATOR}]'
    )

print(f'\nSweep complete — {len(all_results)}/15 categories.')


In [ ]:
# ── Cell 8: Save output ──────────────────────────────────────────────────────
# Writes ONE combined JSON file to /kaggle/working/ (the Kaggle output dir).
# Download this file from the Output tab and drop it into your local results/.
import json as _json
from pathlib import Path

OUT_FILE = Path('/kaggle/working/patchcore_mvtec_results.json')
OUT_FILE.write_text(_json.dumps(all_results, indent=2))

print(f'Output written : {OUT_FILE}')
print(f'File size      : {OUT_FILE.stat().st_size / 1024:.1f} KB')
print(f'Categories     : {len(all_results)}')
print()
print('── Next steps ──────────────────────────────────────────────────────────')
print('1. Click Output tab (right panel) → download patchcore_mvtec_results.json')
print('2. Copy to your local repo:  cp ~/Downloads/patchcore_mvtec_results.json results/')
print('3. Run report cell locally:  generate("results/", "REPORT.md")')
print('4. Commit:  git add results/patchcore_mvtec_results.json && git commit -m ...')


In [ ]:
# ── Cell 9: Summary table ────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(all_results)
for col in ['auroc', 'f1', 'mean_latency_ms']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Model      : {MODEL_ID}')
print(f'Mean AUROC : {df.auroc.mean():.4f}')
print(f'Mean F1    : {df.f1.mean():.4f}')
print(f'Avg time   : {df.mean_latency_ms.mean()/1000:.0f}s / category')
print(f'Total cost : $0.00 (open-source)')
print()
display(df[['category','auroc','f1','mean_latency_ms']]
        .sort_values('auroc', ascending=False)
        .reset_index(drop=True))
